# 02 — S&P 500 Vollständigen Datensatz herunterladen

**Voraussetzung:** Notebook `01_setup_and_test.ipynb` ist komplett grün durchgelaufen.

Was dieses Notebook macht:
1. Holt aktuelle S&P 500 Konstituenten von Wikipedia.
2. Lädt 15 Jahre OHLCV-Daten für jede dieser Aktien (yfinance, ~30–90 s).
3. Lädt den S&P 500 Index selbst (`^GSPC`) als Benchmark.
4. Lädt 10 Makro-Zeitreihen von FRED (VIX, Fed Funds, Yield Curve, CPI, …).
5. Lädt News der letzten 30 Tage für die ersten 50 Tickers (Finnhub Free Tier — voll-S&P 500 dauert wegen Rate-Limit ~10 Minuten und kann separat gestartet werden).
6. Speichert alles als Parquet unter `data/`.

Output-Größe: ~150–250 MB.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from src.data import universe, prices, news, macro
from src.storage import db

## 1. S&P 500 Universum laden

In [ ]:
sp500 = universe.load_sp500(refresh=True)
symbols = universe.symbols()
print(f"Tickers: {len(symbols)}")
print(f"Erste 10: {symbols[:10]}")

## 2. Kursdaten (OHLCV, 15 Jahre)

Ein File pro Ticker unter `data/prices/`.

In [ ]:
cached = prices.download_and_cache(symbols)
print(f"\nErfolgreich gecacht: {len(cached)} / {len(symbols)} Tickers")
missing = sorted(set(symbols) - set(cached))
if missing:
    print(f"Fehlend ({len(missing)}): {missing[:20]}{' ...' if len(missing) > 20 else ''}")

## 3. Benchmark-Index `^GSPC`

In [ ]:
spx = prices.download_one(config.DEFAULT_BENCHMARK)
db.write_prices("_GSPC", spx)  # underscore-prefix => not a real ticker
print(f"S&P 500 Index: {len(spx)} Zeilen, {spx.index.min().date()} -> {spx.index.max().date()}")

## 4. Makrodaten (FRED)

In [ ]:
if config.FRED_API_KEY:
    macro_df = macro.fetch_panel(start="2005-01-01")
    macro_path = config.MACRO_DIR / "fred_panel.parquet"
    macro_df.to_parquet(macro_path)
    print(f"Makro-Panel: {macro_df.shape}, gespeichert nach {macro_path}")
    display(macro_df.tail())
else:
    print("Skip — FRED_API_KEY nicht gesetzt.")

## 5. News (Finnhub)

Free-Tier-Limit: 60 calls/min — wir machen daher 1 Call/Sekunde mit `time.sleep(1.1)`.

**Standard:** nur die ersten 50 Tickers, 30 Tage Historie. Setze `LIMIT = None` für alle 500 — das dauert dann ca. 10 Minuten.

In [ ]:
from datetime import datetime

LIMIT = 50            # set to None for all S&P 500
DAYS_BACK = 30

if config.FINNHUB_API_KEY:
    targets = symbols if LIMIT is None else symbols[:LIMIT]
    news_dict = news.fetch_bulk_company_news(targets, days_back=DAYS_BACK)
    year = datetime.utcnow().year
    saved, total_articles = 0, 0
    for sym, df in news_dict.items():
        if df.empty:
            continue
        db.write_news(sym, year, df)
        saved += 1
        total_articles += len(df)
    print(f"\nNews gespeichert: {saved} Tickers, {total_articles} Artikel insgesamt")
else:
    print("Skip — FINNHUB_API_KEY nicht gesetzt.")

## 6. Sanity-Check — was haben wir?

In [ ]:
price_files = list(config.PRICES_DIR.glob("*.parquet"))
news_files = list(config.NEWS_DIR.glob("*.parquet"))
macro_files = list(config.MACRO_DIR.glob("*.parquet"))

def _mb(files):
    return sum(f.stat().st_size for f in files) / (1024 * 1024)

print(f"Kurs-Files:  {len(price_files):>4}   ({_mb(price_files):.1f} MB)")
print(f"News-Files:  {len(news_files):>4}   ({_mb(news_files):.1f} MB)")
print(f"Makro-Files: {len(macro_files):>4}   ({_mb(macro_files):.1f} MB)")

## Nächste Schritte (Phase 2)

Mit den Daten in `data/` können wir jetzt:
- **Korrelations-Matrix** über alle 500 Aktien rechnen (rolling, z.B. 60 Tage)
- **Technische Indikatoren** (RSI, MACD, Bollinger) mit `ta` ergänzen
- **News-Sentiment** mit FinBERT scoren — das nutzt deine GPUs voll aus
- **Erste Modell-Features** zusammenstellen (Rendite + Sentiment + Makro)

Sag Bescheid welcher dieser Schritte als nächstes kommt.